In [ ]:
%load_ext autoreload
%autoreload 2
import os
import time
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
import torch.nn as nn

from ag_utils import Corpus
from ag_utils import parse_ag_file
from ag_utils import parse_node_properties

from public_data import  prepare_raw_pools, load_graphs_from_pool
import os
import random
import numpy as np
import torch

# >>>>>>>>>> 固定所有随机种子 <<<<<<<<<<
def force_reproducibility(seed=44):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    # 强制 PyTorch 报错或警告非确定性算法
    torch.use_deterministic_algorithms(True, warn_only=True)
    # CUDA 10.1+ 需要这个环境变量
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

force_reproducibility(44)
# >>>>>>>>>> End of seed setting <<<<<<<<<<
from models import NN, GCN,TransformerModel,MainModel,DilatedCNNBranch
from model_utils import train, predict_prob, evaluate_performance
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"


In [ ]:
# parse attack graph file generated by MulVAL tool
attack_graph_path = '../../mulval_attack_graph/ddos_large.dot'
nodes, edges, node_properties = parse_ag_file(attack_graph_path)
node_dict = parse_node_properties(nodes, node_properties)

# save node label into corpus object
corpus = Corpus(node_dict)
num_tokens = corpus.get_num_tokens()
node_features = corpus.get_node_features()
node_types = corpus.get_node_types()
vocab_size = len(corpus.dictionary)
print('vocab_size: ', vocab_size)
print('num_tokens: ', num_tokens)
print('node_features shape: ', node_features.shape)


In [ ]:
# statistics of the encoded attack graph
num_nodes = len(nodes)
print('num_nodes: ', num_nodes)

num_node_features = node_features.shape[1]
print('num_node_features: ', num_node_features)

num_edges = len(edges)
print('num_edges: ', num_edges)

action_nodes = corpus.get_action_nodes()
action_node_idx = list(action_nodes.keys())
num_action_nodes = len(action_node_idx)
print('action_node_idx: ', action_node_idx)
print('num_action_nodes: ', num_action_nodes)

# var 'action_mask' is used to represent the attack scenarios in attack graph (i.e., the privilege nodes)
action_mask = action_node_idx


In [ ]:
# adj matrix and edge index
adj_matrix = torch.zeros(len(nodes), len(nodes))

for edge in edges:
    source_node, target_node = edge
    source_index = nodes.index(source_node)
    target_index = nodes.index(target_node)
    adj_matrix[source_index][target_index] = 1

edge_index = adj_matrix.nonzero().t().contiguous()

assert edge_index.shape[0]==2

In [ ]:
# 1. 预切分流量池 (关键：物理隔离)
pools = prepare_raw_pools('../../datasets/CIC-IDS2017.csv')
force_reproducibility(44)
# 2. 分别构造训练、验证、测试图集
# 训练集可以多造点，验证/测试集按比例减少
x_b_train, y_b_train, x_m_train, y_m_train = load_graphs_from_pool(pools['train'], 1500, 500, action_node_idx)
x_b_val, y_b_val, x_m_val, y_m_val = load_graphs_from_pool(pools['val'], 200, 50, action_node_idx)
x_b_test, y_b_test, x_m_test, y_m_test = load_graphs_from_pool(pools['test'], 200, 50, action_node_idx)

# 3. 辅助函数：将 load 出来的结果转为最终图格式 (代码复用你之前的 gene_dataset 逻辑)
def build_final_tensor(x_b, y_b, x_m, y_m, num_nodes, action_node_idx):
    force_reproducibility(44)
    rt_meas_dim = x_b.shape[2]
    # 合并良性和恶意
    X = torch.cat([
        torch.zeros(x_b.shape[0], num_nodes, rt_meas_dim).index_copy_(1, torch.tensor(action_node_idx), x_b),
        torch.zeros(x_m.shape[0], num_nodes, rt_meas_dim).index_copy_(1, torch.tensor(action_node_idx), x_m)
    ])
    Y = torch.cat([y_b, y_m])
    return X, Y

X_train_raw, Y_train = build_final_tensor(x_b_train, y_b_train, x_m_train, y_m_train, num_nodes, action_node_idx)
X_val_raw, Y_val = build_final_tensor(x_b_val, y_b_val, x_m_val, y_m_val, num_nodes, action_node_idx)
X_test_raw, Y_test = build_final_tensor(x_b_test, y_b_test, x_m_test, y_m_test, num_nodes, action_node_idx)

# 4. 归一化 (Scaler 必须只 fit 训练集)
scaler = MinMaxScaler()
X_train_reshaped = X_train_raw.reshape(-1, X_train_raw.shape[-1])
scaler.fit(X_train_reshaped)

def scale_and_tensor(X_raw, scaler):
    N, N_nodes, Feats = X_raw.shape
    X_scaled = scaler.transform(X_raw.reshape(-1, Feats)).reshape(N, N_nodes, Feats)
    return torch.tensor(X_scaled, dtype=torch.float32)

X_train = scale_and_tensor(X_train_raw, scaler)

X_val = scale_and_tensor(X_val_raw, scaler)
X_test = scale_and_tensor(X_test_raw, scaler)

In [ ]:

# hyperparameters for training
in_dim = X_train.shape[-1]
hidden_dim = 32
out_dim = 1
lr = 0.001
device = 'cuda'
# model initialization
models = {}


model_MainModel=MainModel(in_dim, hidden_dim, out_dim,4)
model_MainModel.action_mask = action_mask
WithoutGCN=MainModel(in_dim, hidden_dim, out_dim,1)
WithoutGCN.action_mask = action_mask
WithoutTransformer=MainModel(in_dim, hidden_dim, out_dim,2)
WithoutTransformer.action_mask = action_mask
WithoutCNN=MainModel(in_dim, hidden_dim, out_dim,3)
WithoutCNN.action_mask = action_mask
model_transformer=TransformerModel(in_dim,hidden_dim,out_dim)
#model_NN = NN(78, hidden_dim, out_dim)
model_GCN = GCN(in_dim, hidden_dim, out_dim)
model_DCNN=DilatedCNNBranch(78,hidden_dim,out_dim)
models['mainModel']=model_MainModel
models['WithoutGCN']=WithoutGCN
models['WithoutTransformer']=WithoutTransformer
models['WithoutCNN']=WithoutCNN
models['DCNN']=model_DCNN
models['transformer'] = model_transformer
models['GCN'] = model_GCN




for name, model in models.items():
    model.name = name
    model.action_mask = action_mask
    num_epochs = 50 # early stop when overfitting observed

    print(f'{model.name} start training {num_epochs} epochs')
    time_start = time.time()
    print('model: ', model) 
    train(model, lr, num_epochs, X_train, Y_train, X_val, Y_val, edge_index, 78, device)
    time_end = time.time()
    print('time cost: ', time_end - time_start)  
    print(f'{model.name} training finished!')

    print(f'{model.name} accuracy on training set: {model.stat["acc_train"][-1]}')
    print(f'{model.name} accuracy on validation set: {model.stat["acc_val"][-1]}')
    print()
    print()


In [ ]:
#plot the roc curve
from sklearn.metrics import roc_curve, auc
fig, ax = plt.subplots(figsize=(5, 5))
for name, model in models.items():
    prob = predict_prob(model, X_test, edge_index, 78,device='cuda')  # 或 'cpu'

    y_true = Y_test.view(-1).cpu().numpy()
    y_score = prob.view(-1, 2)[:, 1].cpu().numpy()
    print(f"{name} - y_true samples: {len(y_true)}, y_score samples: {len(y_score)}")
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label='{} (AUC = {:.4f})'.format(model.name, roc_auc))
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend()
plt.show()


In [ ]:
metrics = evaluate_performance(models, X_test, Y_test, edge_index, device)
df = pd.DataFrame(metrics)
print(df)

In [ ]:
# plot the TP, FP, TN, FN for each model
fig, axs = plt.subplots(2, 2, figsize=(5, 5))
axs = axs.ravel()

bar_width = 0.5
labels = ['TN', 'FP', 'FN', 'TP']

for i, label in enumerate(labels):
    for j, name in enumerate(models):
        rects = axs[i].bar(j, metrics[j][label], width=bar_width, label=name)
    axs[i].set_xticks(np.arange(len(models)))
    axs[i].set_xticklabels(models.keys())
    for tick in axs[i].xaxis.get_major_ticks():
        tick.label1.set_fontsize(10)
    axs[i].set_title(label)

handles, labels = axs[0].get_legend_handles_labels()
# fig.legend(handles, labels, loc='upper right', bbox_to_anchor=(1.2, 1))

plt.tight_layout()
plt.show()



In [ ]:
# robustness evaluation
for i in range(Y_test.shape[0]):
    for j in range(len(action_mask)):
        if Y_test[i, j] == 1:
            for k in range(78):
                X_test[i, action_mask[j],-78+k] += torch.normal(mean=0, std=0.01, size=(1,)).item()

metrics = evaluate_performance(models, X_test, Y_test, edge_index, device)
df = pd.DataFrame(metrics)
print(df)

In [ ]:
from torch_geometric.data import Data
from torch_geometric.explain import GNNExplainer, Explainer

# individual case analysis
found = False
while not found:
    indi_case_id = torch.randint(0, X_test.shape[0], (1,)).item()
    y_true = Y_test[indi_case_id]
    if y_true[2] != 0:
        found = True
        print('indi_case_id: ', indi_case_id)
indi_case_x = X_test[indi_case_id]

In [ ]:
# Explainability visualization
#
# model_to_explain = model_GCN_EW
# model_to_explain.eval()
# if model.name in ['GAT']:
#     indi_case_x = indi_case_x.view(-1, in_dim)
# prob_ts = torch.sigmoid(model_to_explain(indi_case_x, edge_index))[action_mask]
#
# print('model prob_ts: \n', prob_ts)
# print('y_true: ', y_true)
#
# data = Data(x=X_test[indi_case_id], edge_index=edge_index)
#
# explainer = Explainer(
#     model=model_to_explain,
#     algorithm=GNNExplainer(epochs=200),
#     explanation_type='model',
#     node_mask_type='attributes',
#     edge_mask_type='object',
#     model_config=dict(
#         mode='binary_classification',
#         task_level='node',
#         return_type='raw',
#     ),
#
# )
#
# feat_labels_ag = corpus.dictionary.idx2word.values()
# feat_labels_ag = list(feat_labels_ag)
# feat_labels_rtm = pd.read_csv('../../datasets/CICIDS-2017.csv').columns.tolist()[:-1]
# feat_labels_rtm = [feat.strip() for feat in feat_labels_rtm]
# feat_labels = feat_labels_ag + feat_labels_rtm
#
# compromised_node = action_node_idx[np.where(y_true==1)[0].item()]
# explanation = explainer(data.x, data.edge_index, index=compromised_node)
#
# explanation.visualize_feature_importance(feat_labels=feat_labels, top_k=10)
# explanation.visualize_graph()